In [26]:
from utils import (
    read_intensities,
    scan_motif_kmers,
    read_unique_kmer_positions,
    get_sequence_from_fasta,
    print_rows_as_tsv,
    plot_aff_motif_effects,
    run_snv_regression,
    reverse_complement
)

# genome =  "/home/aki/repos/INVPBM/rust_INVPBM/data/genome/hg38_ucsc.fa"
genome = "../../../../../d/OneDrive - McGill University/repos/hg38_ucsc.fa"
intensities = read_intensities("data/intensities/GABPA_MCF7_probeIntensity.bed")
kmers = read_unique_kmer_positions("data/array/MCF7_Array_8mer_ATAC.txt")
# intensities = read_intensities("data/intensities/DNase_GABPA_MCF7_ENCFF835KCG_probeIntensity.bed")
# kmers = read_unique_kmer_positions("data/array/MCF7_Array_8mer_DNase.txt")
mode = "neg-binomial"
kmer_size=8

In [75]:
region = "chr5:1295108-1295118"
# region = "chr5:1295105-1295140"
chrom, coords = region.split(":")
start, end = map(int, coords.split("-"))
region_seq = get_sequence_from_fasta(chrom, start, end, genome)
region_seq

'GGCCCGGAGGGGGCTGGGCCGGGGACCCGGGAGGGG'

In [52]:
allele_region_offsets, allele_matched_kmers = scan_motif_kmers(
        region_seq=region_seq,
        kmer_size=kmer_size,
        kmer_positions=kmers,
    )

In [4]:
# from collections import defaultdict

# def scan_motif_kmers(region_seq, kmer_size, kmer_positions):
#     """
#     Slides a k-mer window across the sequence and builds:
#     - allele_region_offsets[motif_pos][snv_index][allele][region_id] = offset
#     - allele_matched_kmers[motif_pos][snv_index][allele] = list of (wildcard_kmer, matched_kmer, region_id)

#     Returns:
#         (allele_region_offsets, allele_matched_kmers)
#     """

#     allele_region_offsets = defaultdict(lambda: defaultdict(lambda: defaultdict(dict)))
#     allele_matched_kmers = defaultdict(lambda: defaultdict(lambda: defaultdict(list)))

#     seen = set()  # prevent duplicate matches per motif_pos + snv + base + region + kmer

#     for motif_pos in range(len(region_seq) - kmer_size + 1):
#         kmer = region_seq[motif_pos:motif_pos + kmer_size]

#         for snv_index in range(kmer_size):
#             # Create wildcarded k-mer
#             wildcard_kmer = list(kmer)
#             wildcard_kmer[snv_index] = "."
#             wildcard_kmer = "".join(wildcard_kmer)

#             for base in "ACGT":
#                 filled_kmer = list(kmer)
#                 filled_kmer[snv_index] = base
#                 filled_kmer = "".join(filled_kmer)

#                 if filled_kmer in kmer_positions:
#                     for region_id, offset in kmer_positions[filled_kmer].items():
#                         key = (motif_pos, snv_index, base, region_id, filled_kmer)
#                         if key in seen:
#                             continue
#                         seen.add(key)

#                         allele_region_offsets[motif_pos][snv_index][base][region_id] = offset
#                         allele_matched_kmers[motif_pos][snv_index][base].append(
#                             (wildcard_kmer, filled_kmer, region_id)
#                         )

#     return allele_region_offsets, allele_matched_kmers



allele_region_offsets, allele_matched_kmers = scan_motif_kmers(
    region_seq=region_seq,
    kmer_size=kmer_size,
    kmer_positions=kmers,
)

In [ ]:
from collections import defaultdict

kmer_counts = defaultdict(int)
seen = set()

for motif_pos in allele_matched_kmers:
    for snv_index in allele_matched_kmers[motif_pos]:
        for allele in allele_matched_kmers[motif_pos][snv_index]:
            for wildcard_kmer, full_kmer, region_id in allele_matched_kmers[motif_pos][snv_index][allele]:
                key = (full_kmer, region_id)
                if key not in seen:
                    seen.add(key)
                    kmer_counts[full_kmer] += 1

# Print results sorted by descending count
for kmer, count in sorted(kmer_counts.items(), key=lambda x: -x[1]):
    print(f"{kmer}\t{count}")

# ABOVE WORKS



In [76]:
from collections import defaultdict

def scan_motif_kmers(region_seq, kmer_size, kmer_positions):
    """
    Slides a k-mer window across the sequence and builds:
    - allele_region_offsets[motif_pos][snv_index][allele][region_id] = offset
    - allele_matched_kmers[motif_pos][snv_index][allele] = list of (wildcard_kmer, matched_kmer, region_id)

    Returns:
        (allele_region_offsets, allele_matched_kmers)
    """

    allele_region_offsets = defaultdict(lambda: defaultdict(lambda: defaultdict(dict)))
    allele_matched_kmers = defaultdict(lambda: defaultdict(lambda: defaultdict(list)))

    seen = set()  # prevent duplicate matches per motif_pos + snv + base + region + kmer

    for motif_pos in range(len(region_seq) - kmer_size + 1):
        kmer = region_seq[motif_pos:motif_pos + kmer_size]

        for snv_index in range(kmer_size):
            # Create wildcarded k-mer
            wildcard_kmer = list(kmer)
            wildcard_kmer[snv_index] = "."
            wildcard_kmer = "".join(wildcard_kmer)

            for base in "ACGT":
                filled_kmer = list(kmer)
                filled_kmer[snv_index] = base
                filled_kmer = "".join(filled_kmer)
                filled_kmer_reverse_complement = reverse_complement(filled_kmer)

                for match_kmer in [filled_kmer, filled_kmer_reverse_complement]:
                    if match_kmer in kmer_positions:
                        for region_id, offset in kmer_positions[match_kmer].items():
                            key = (motif_pos, snv_index, base, region_id, match_kmer)
                            if key in seen:
                                continue
                            seen.add(key)

                            allele_region_offsets[motif_pos][snv_index][base][region_id] = offset

                            # Always store the forward-facing version (filled_kmer)
                            allele_matched_kmers[motif_pos][snv_index][base].append(
                                (wildcard_kmer, filled_kmer, region_id)
                            )
                            
                            # Always store both forward and reverse-facing version (match_kmer)
                            # allele_matched_kmers[motif_pos][snv_index][base].append(
                            #     (wildcard_kmer, match_kmer, region_id)
                            # )
    
    return allele_region_offsets, allele_matched_kmers


allele_region_offsets, allele_matched_kmers = scan_motif_kmers(
    region_seq=region_seq,
    kmer_size=kmer_size,
    kmer_positions=kmers,
)

In [22]:
def print_kmer_counts_by_position(allele_matched_kmers):
    for motif_pos in sorted(allele_matched_kmers):
        for snv_index in sorted(allele_matched_kmers[motif_pos]):
            kmer_counts = defaultdict(int)
            for allele in allele_matched_kmers[motif_pos][snv_index]:
                for _, full_kmer, _ in allele_matched_kmers[motif_pos][snv_index][allele]:
                    kmer_counts[full_kmer] += 1

            if kmer_counts:
                total = sum(kmer_counts.values())
                print(f"motif_pos: {motif_pos}, snv_index: {snv_index} (total: {total})")
                for kmer, count in sorted(kmer_counts.items(), key=lambda x: -x[1]):
                    print(f"  {kmer}\t{count}")
                print()

print_kmer_counts_by_position(allele_matched_kmers)

motif_pos: 0, snv_index: 0 (total: 2977)
  GCGGAGGG	1241
  CCGGAGGG	932
  ACGGAGGG	443
  TCGGAGGG	361

motif_pos: 0, snv_index: 1 (total: 8058)
  CTGGAGGG	2898
  CAGGAGGG	2649
  CGGGAGGG	1579
  CCGGAGGG	932

motif_pos: 0, snv_index: 2 (total: 8366)
  CCAGAGGG	3711
  CCTGAGGG	2801
  CCGGAGGG	932
  CCCGAGGG	922

motif_pos: 0, snv_index: 3 (total: 2482)
  CCGCAGGG	975
  CCGGAGGG	932
  CCGAAGGG	403
  CCGTAGGG	172

motif_pos: 0, snv_index: 4 (total: 2720)
  CCGGAGGG	932
  CCGGCGGG	905
  CCGGTGGG	478
  CCGGGGGG	405

motif_pos: 0, snv_index: 5 (total: 2270)
  CCGGAGGG	932
  CCGGAAGG	814
  CCGGATGG	278
  CCGGACGG	246

motif_pos: 0, snv_index: 6 (total: 2648)
  CCGGAGGG	932
  CCGGAGAG	678
  CCGGAGCG	619
  CCGGAGTG	419

motif_pos: 0, snv_index: 7 (total: 3113)
  CCGGAGGC	1080
  CCGGAGGG	932
  CCGGAGGA	697
  CCGGAGGT	404

motif_pos: 1, snv_index: 0 (total: 10700)
  GGGAGGGG	5335
  AGGAGGGG	2853
  TGGAGGGG	1607
  CGGAGGGG	905

motif_pos: 1, snv_index: 1 (total: 4391)
  CAGAGGGG	1608
  CTGAGGGG	124

In [23]:
from collections import defaultdict

motif_pos = 0
snv_index = 0

alleles_all = allele_region_offsets[motif_pos][snv_index]
region_allele_hits = defaultdict(set)

# Step 1: Build full allele-to-region mapping (including ref)
for allele, region_map in alleles_all.items():
    for region in region_map:
        region_allele_hits[region].add(allele)

# Step 2: Classify regions
ambiguous = []
unambiguous = []
has_intensity = []
used_by_python = []

for region, alleles in region_allele_hits.items():
    if len(alleles) > 1:
        ambiguous.append(region)
    else:
        unambiguous.append(region)
    if region in intensities:
        has_intensity.append(region)
    if len(alleles) == 1 and region in intensities and list(alleles)[0] != region_seq[motif_pos + snv_index]:
        used_by_python.append(region)

# Step 3: Print diagnostics
print(f"Total regions (all alleles): {len(region_allele_hits)}")
print(f" - Ambiguous (multi-allele): {len(ambiguous)}")
print(f" - Unambiguous: {len(unambiguous)}")
print(f" - Have intensity: {len(has_intensity)}")
print(f" - Usable by Python (non-ref, unambiguous, intensity): {len(used_by_python)}")
print(f" - Would be used by Perl (unambiguous + intensity): {len([r for r in unambiguous if r in intensities])}")


Total regions (all alleles): 2865
 - Ambiguous (multi-allele): 95
 - Unambiguous: 2770
 - Have intensity: 2865
 - Usable by Python (non-ref, unambiguous, intensity): 1910
 - Would be used by Perl (unambiguous + intensity): 2770


In [10]:
def query_kmer_and_reverse(kmer, kmer_positions):
    rc = reverse_complement(kmer)
    
    count_fwd = len(kmer_positions.get(kmer, {}))
    count_rev = len(kmer_positions.get(rc, {}))
    count_total = count_fwd + count_rev

    print(f"Query: {kmer}")
    print(f"Forward ({kmer}): {count_fwd}")
    print(f"Reverse ({rc}): {count_rev}")
    print(f"Total: {count_total}")

    return count_fwd, count_rev, count_total

query_kmer_and_reverse("GGGAGGGG", kmers)


Query: GGGAGGGG
Forward (GGGAGGGG): 8558
Reverse (CCCCTCCC): 8461
Total: 17019


(8558, 8461, 17019)

In [52]:
from pprint import pprint

motif_pos = 0
snv_index = 0

# 0.1 Reference allele from region_seq
ref_allele = region_seq[motif_pos + snv_index]
print(f"Reference allele at pos {motif_pos + snv_index}: {ref_allele}")

# 0.2 All alleles at this motif/snv position
all_alleles = allele_region_offsets[motif_pos][snv_index]
print(f"All alleles: {sorted(all_alleles.keys())}")

# 0.3 Filter non-reference alleles
alt_alleles = {a: all_alleles[a] for a in all_alleles if a != ref_allele}
print(f"Modeled alleles (excluding ref): {sorted(alt_alleles.keys())}")

# 1.1 Gather all region IDs per allele
allele_to_regions = {a: set(alt_alleles[a].keys()) for a in alt_alleles}
all_regions = set.union(*allele_to_regions.values())

print(f"Total unique regions (non-ref alleles): {len(all_regions)}")

# 1.2 Count how many alleles each region maps to
region_counts = defaultdict(int)
for allele, regions in allele_to_regions.items():
    for region in regions:
        region_counts[region] += 1

# 1.3 Classify ambiguous and unambiguous
ambiguous = [r for r, count in region_counts.items() if count > 1]
unambiguous = [r for r, count in region_counts.items() if count == 1]

print(f"Ambiguous regions: {len(ambiguous)}")
print(f"Unambiguous regions: {len(unambiguous)}")

# 1.4 See a few examples
print("Example ambiguous regions:", ambiguous[:5])
print("Example unambiguous regions:", unambiguous[:5])


Reference allele at pos 0: C
All alleles: ['A', 'C', 'G', 'T']
Modeled alleles (excluding ref): ['A', 'G', 'T']
Total unique regions (non-ref alleles): 7668
Ambiguous regions: 631
Unambiguous regions: 7037
Example ambiguous regions: ['chr8:730628-731852', 'chr15:80403693-80405235', 'chr13:50130281-50130913', 'chr1:151596304-151597477', 'chr11:35418752-35419848']
Example unambiguous regions: ['chr7:75358597-75359675', 'chr20:52578568-52578853', 'chr12:116658344-116659103', 'chr2:69893639-69894679', 'chr5:143042057-143042267']


In [58]:
from collections import defaultdict

motif_pos = 0
snv_index = 0
ref_allele = region_seq[motif_pos + snv_index]

## Get all alleles at this position
# all_alleles = allele_region_offsets[motif_pos][snv_index]

# # Filter out the reference allele
# alleles = {a: v for a, v in all_alleles.items() if a != ref_allele}

# # Build region -> allele mapping
# region_to_alleles = defaultdict(set)
# for allele, region_dict in alleles.items():
#     for region in region_dict:
#         region_to_alleles[region].add(allele)

# # Filter to unambiguous regions that map to exactly one non-ref allele and have intensity
# unambiguous_regions = [
#     region for region, allele_set in region_to_alleles.items()
#     if len(allele_set) == 1 and region in intensities
# ]
# Add this for completeness:
alleles = {a: v for a, v in all_alleles.items()}  # include ref too!

# Track all allele assignments
region_to_alleles = defaultdict(set)
for allele, region_dict in alleles.items():
    for region in region_dict:
        region_to_alleles[region].add(allele)

# Keep unambiguous regions (only one allele hit)
unambiguous_regions = [
    region for region, allele_set in region_to_alleles.items()
    if len(allele_set) == 1 and region in intensities
]


print(f"[Step 2] Found {len(unambiguous_regions)} unambiguous, non-ref regions with intensity")

# Show some example regions and their mapped allele
for region in unambiguous_regions[:10]:
    allele = list(region_to_alleles[region])[0]
    print(f"  {region} → {allele} (intensity: {intensities[region]})")


[Step 2] Found 8783 unambiguous, non-ref regions with intensity
  chr1:10837413-10837625 → A (intensity: 1.260089993)
  chr1:108595390-108595910 → A (intensity: 0.877030015)
  chr1:109282684-109283845 → A (intensity: 21.25737953)
  chr1:110412535-110412872 → A (intensity: 2.192369938)
  chr1:110963741-110964402 → A (intensity: 95.31704712)
  chr1:114248536-114248879 → A (intensity: 1.883780003)
  chr1:114511781-114512334 → A (intensity: 2.152529955)
  chr1:116219734-116220509 → A (intensity: 1.674780011)
  chr1:116708244-116708885 → A (intensity: 2.382610083)
  chr1:117121334-117122654 → A (intensity: 4.269249916)


In [63]:
# # Step 3: Build design matrix X and response vector y
# design = []
# y_values = []
# region_list = []

# for region in unambiguous_regions:
#     row = {a: 0 for a in alleles}
#     assigned_allele = list(region_to_alleles[region])[0]
#     row[assigned_allele] = 1
#     design.append(row)
#     y_values.append(intensities[region])
#     region_list.append(region)

# X = pd.DataFrame(design, index=region_list)
# y = pd.Series(y_values, index=region_list)

# print("[Step 3] Design matrix shape:", X.shape)
# print("[Step 3] y shape:", y.shape)

# # Display summary stats
# display(X.head())
# print(y.describe())


# Step 3: Build design matrix X and response vector y
design = []
y_values = []
region_list = []

# Get non-reference alleles only (to use in design matrix)
non_ref_alleles = [a for a in alleles if a != ref_allele]

for region in unambiguous_regions:
    assigned_allele = list(region_to_alleles[region])[0]

    # Skip regions where the allele is not in the expected set (just in case)
    if assigned_allele not in non_ref_alleles + [ref_allele]:
        continue

    # One-hot encode: reference allele is all zeros
    row = {a: 0 for a in non_ref_alleles}
    if assigned_allele != ref_allele:
        row[assigned_allele] = 1

    design.append(row)
    y_values.append(intensities[region])
    region_list.append(region)

# Create design matrix and response vector
X = pd.DataFrame(design, index=region_list)
y = pd.Series(y_values, index=region_list)

print("[Step 3] Design matrix shape:", X.shape)
print("[Step 3] y shape:", y.shape)

# Display summary stats
display(X.head())
print(y.describe())


[Step 3] Design matrix shape: (8783, 3)
[Step 3] y shape: (8783,)


,A,G,T
chr1:10837413-10837625,1,0,0
chr1:108595390-108595910,1,0,0
chr1:109282684-109283845,1,0,0
chr1:110412535-110412872,1,0,0
chr1:110963741-110964402,1,0,0


count    8783.000000
mean        8.337662
std        14.869135
min         0.000000
25%         2.126240
50%         3.157130
75%         5.788000
max       156.466492
dtype: float64


In [64]:
include_covariates = True
# Step 4: Add covariates (if enabled)
if include_covariates:
    kmer_pos = {
        region: allele_region_offsets[motif_pos][snv_index][allele][region]
        for allele in alleles
        for region in allele_region_offsets[motif_pos][snv_index][allele]
        if region in region_list
    }

    lp, sl = extract_covariates(region_list, kmer_pos)
    assert len(lp) == len(sl) == len(region_list), "Covariate mismatch!"

    X["lp"] = lp
    X["sl"] = sl

    print("[Step 4] Added covariates:")
    print(X[["lp", "sl"]].describe())


[Step 4] Added covariates:
                lp           sl
count  8783.000000  8783.000000
mean      0.500018   955.874758
std       0.259301   445.851617
min       0.000598   152.000000
25%       0.290147   610.000000
50%       0.499114   949.000000
75%       0.710120  1249.000000
max       0.996000  2984.000000


In [65]:
# Step 5: Fit regression model
X_const = sm.add_constant(X)
# y_rounded = y.round().astype(int)
model = sm.GLM(y, X_const, family=sm.families.NegativeBinomial())
results = model.fit()

print("[Step 5] Condition number:", np.linalg.cond(X_const.values))
print("[Step 5] Fitted coefficients:")
print(results.params)


[Step 5] Condition number: 5075.828425209938
[Step 5] Fitted coefficients:
const    0.846337
A       -0.116830
G       -0.045223
T       -0.022768
lp      -0.054977
sl       0.001278
dtype: float64


In [66]:
print(results.summary())

                 Generalized Linear Model Regression Results                  
Dep. Variable:                      y   No. Observations:                 8783
Model:                            GLM   Df Residuals:                     8777
Model Family:        NegativeBinomial   Df Model:                            5
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -26974.
Date:                Thu, 24 Apr 2025   Deviance:                       9009.5
Time:                        10:19:54   Pearson chi2:                 2.23e+04
No. Iterations:                    10   Pseudo R-squ. (CS):             0.1932
Covariance Type:            nonrobust                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.8463      0.040     21.375      0.0

In [78]:
# def run_snv_regression(
#     motif_pos,
#     snv_index,
#     allele_region_offsets,
#     allele_matched_kmers,
#     intensities,
#     region_seq=None,  
#     model_type="nb",
#     include_covariates=True,
# ):
#     rows = []

#     # Determine reference allele from the original sequence
#     ref_allele = region_seq[motif_pos + snv_index]
#     all_alleles = allele_region_offsets[motif_pos][snv_index]
#     alleles = {a: v for a, v in all_alleles.items() if a != ref_allele}

#     all_regions = set()
#     for region_dict in alleles.values():
#         all_regions.update(region_dict.keys())

#     design = []
#     y_values = []
#     region_list = []

#     for region in all_regions:
#         row = {a: 0 for a in alleles}
#         for a in alleles:
#             if region in alleles[a]:
#                 row[a] = 1

#         # Exclude ambiguous regions that hit >1 allele
#         if sum(row.values()) == 1 and region in intensities:
#             design.append(row)
#             y_values.append(intensities[region])
#             region_list.append(region)

#     if not design:
#         return []

#     X = pd.DataFrame(design)
#     y = pd.Series(y_values, index=region_list)
#     X.index = y.index

#     if include_covariates:
#         kmer_pos = {
#             region: allele_region_offsets[motif_pos][snv_index][allele][region]
#             for allele in alleles
#             for region in allele_region_offsets[motif_pos][snv_index][allele]
#         }
#         lp, sl = extract_covariates(region_list, kmer_pos)
#         X["lp"] = lp
#         X["sl"] = sl

#     X_const = sm.add_constant(X)

#     if model_type == "ols":
#         model = sm.OLS(y.apply(math.log1p), X_const)
#     else:
#         model = sm.GLM(y, X_const, family=sm.families.NegativeBinomial())

#     results = model.fit()

#     seen = set()
#     for allele in alleles:
#         coef = results.params.get(allele, math.nan)
#         pval = results.pvalues.get(allele, math.nan)

#         for wildcard_kmer, filled_kmer, _ in allele_matched_kmers[motif_pos][snv_index][allele]:
#             key = (wildcard_kmer, filled_kmer)
#             if key in seen:
#                 continue
#             seen.add(key)

#             row = [
#                 wildcard_kmer,
#                 filled_kmer,
#                 motif_pos,
#                 snv_index,
#                 "AFF",
#                 allele,
#                 coef,
#                 pval,
#                 motif_pos + snv_index,
#             ]
#             rows.append(row)

#     # Add reference allele with NA effect and NA pval
#     for wildcard_kmer, filled_kmer, _ in allele_matched_kmers[motif_pos][snv_index].get(ref_allele, []):
#         key = (wildcard_kmer, filled_kmer)
#         if key in seen:
#             continue
#         seen.add(key)
#         row = [
#             wildcard_kmer,
#             filled_kmer,
#             motif_pos,
#             snv_index,
#             "AFF",
#             ref_allele,
#             "NA",
#             "NA",
#             motif_pos + snv_index,
#         ]
#         rows.append(row)

#     return rows

def run_snv_regression(
    motif_pos,
    snv_index,
    allele_region_offsets,
    allele_matched_kmers,
    intensities,
    region_seq=None,
    model_type="nb",
    include_covariates=True,
):
    import pandas as pd
    import numpy as np
    import math
    import statsmodels.api as sm
    from collections import defaultdict
    from utils import extract_covariates  # Ensure your utils has this

    rows = []

    # [Step 1] Reference allele
    ref_allele = region_seq[motif_pos + snv_index]
    all_alleles = allele_region_offsets[motif_pos][snv_index]
    alleles = sorted(set(all_alleles.keys()))
    alt_alleles = [a for a in alleles if a != ref_allele]

    # print(f"[Step 1] Ref allele: {ref_allele}")
    # print(f"[Step 1] Modeled alleles: {alt_alleles}")

    # [Step 2] Build region → allele mapping
    region_to_alleles = defaultdict(set)
    for allele, regions in all_alleles.items():
        for region in regions:
            region_to_alleles[region].add(allele)

    unambiguous_regions = [
        region for region, allele_set in region_to_alleles.items()
        if len(allele_set) == 1 and region in intensities
    ]

    # print(f"[Step 2] Found {len(unambiguous_regions)} usable regions (unambiguous + intensity)")

    # [Step 3] Build X and y
    design = []
    y_values = []
    region_list = []

    for region in unambiguous_regions:
        assigned_allele = list(region_to_alleles[region])[0]
        row = {a: 0 for a in alt_alleles}  # exclude ref
        if assigned_allele in alt_alleles:
            row[assigned_allele] = 1
        design.append(row)
        y_values.append(intensities[region])
        region_list.append(region)

    if not design:
        return []

    X = pd.DataFrame(design, index=region_list)
    y = pd.Series(y_values, index=region_list)

    # [Step 4] Add covariates
    if include_covariates:
        kmer_pos = {
            region: allele_region_offsets[motif_pos][snv_index][allele][region]
            for allele in alleles
            for region in allele_region_offsets[motif_pos][snv_index][allele]
            if region in region_list
        }
        lp, sl = extract_covariates(region_list, kmer_pos)
        X["lp"] = lp
        X["sl"] = sl

    # [Step 5] Fit model
    X_const = sm.add_constant(X)
    if model_type == "ols":
        model = sm.OLS(y.apply(math.log1p), X_const)
    else:
        model = sm.GLM(y, X_const, family=sm.families.NegativeBinomial())

    results = model.fit()

    # [Step 6] Output results
    seen = set()
    for allele in alt_alleles:
        coef = results.params.get(allele, math.nan)
        pval = results.pvalues.get(allele, math.nan)

        for wildcard_kmer, filled_kmer, _ in allele_matched_kmers[motif_pos][snv_index].get(allele, []):
            key = (wildcard_kmer, filled_kmer)
            if key in seen:
                continue
            seen.add(key)
            row = [
                wildcard_kmer,
                filled_kmer,
                motif_pos,
                snv_index,
                "AFF",
                allele,
                coef,
                pval,
                motif_pos + snv_index,
            ]
            rows.append(row)

    # Reference allele gets NA effect
    for wildcard_kmer, filled_kmer, _ in allele_matched_kmers[motif_pos][snv_index].get(ref_allele, []):
        key = (wildcard_kmer, filled_kmer)
        if key in seen:
            continue
        seen.add(key)
        row = [
            wildcard_kmer,
            filled_kmer,
            motif_pos,
            snv_index,
            "AFF",
            ref_allele,
            "NA",
            "NA",
            motif_pos + snv_index,
        ]
        rows.append(row)

    return rows




import re
from collections import defaultdict
import random
import numpy as np
import statsmodels.api as sm
from pyfaidx import Fasta
import warnings
from statsmodels.discrete.discrete_model import NegativeBinomial
import hashlib
import pandas as pd
import math
import matplotlib.pyplot as plt
import sys
from utils import (
    extract_covariates
)

motif_pos = 0
snv_index = 0

rows = run_snv_regression(
    motif_pos,
    snv_index,
    allele_region_offsets,
    allele_matched_kmers,
    intensities,
    region_seq=region_seq,
    include_covariates = True
)


for row in rows:
    print(row)

['.GCCCGGA', 'AGCCCGGA', 0, 0, 'AFF', 'A', -0.15815798395206887, 1.088533115959818e-06, 0]
['.GCCCGGA', 'CGCCCGGA', 0, 0, 'AFF', 'C', 0.1061041676930918, 0.0020268882657664593, 0]
['.GCCCGGA', 'TGCCCGGA', 0, 0, 'AFF', 'T', -0.2171485557948205, 7.48926593570001e-10, 0]
['.GCCCGGA', 'GGCCCGGA', 0, 0, 'AFF', 'G', 'NA', 'NA', 0]


In [80]:
def run_snv_regression(
    motif_pos,
    snv_index,
    allele_region_offsets,
    allele_matched_kmers,
    intensities,
    region_seq=None,
    model_type="nb",
    include_covariates=True,
):
    import pandas as pd
    import numpy as np
    import math
    import statsmodels.api as sm
    from collections import defaultdict
    import time
    from utils import extract_covariates  # assumes your extract_covariates is defined

    start = time.time()
    rows = []

    # [Step 1] Get alleles and reference
    ref_allele = region_seq[motif_pos + snv_index]
    region_lookup = allele_region_offsets[motif_pos][snv_index]
    all_alleles = sorted(region_lookup.keys())
    alt_alleles = [a for a in all_alleles if a != ref_allele]

    # [Step 2] Build region → allele map
    region_to_alleles = defaultdict(set)
    for allele, regions in region_lookup.items():
        for region in regions:
            region_to_alleles[region].add(allele)

    # [Step 3] Select unambiguous regions with intensity
    unambiguous_regions = [
        region for region, allele_set in region_to_alleles.items()
        if len(allele_set) == 1 and region in intensities
    ]
    if not unambiguous_regions:
        return []

    # [Step 4] Build design matrix with numpy
    allele_index = {a: i for i, a in enumerate(alt_alleles)}
    design_array = np.zeros((len(unambiguous_regions), len(alt_alleles)), dtype=np.uint8)
    y_values = []
    region_list = []

    for i, region in enumerate(unambiguous_regions):
        assigned = next(iter(region_to_alleles[region]))
        if assigned in allele_index:
            design_array[i, allele_index[assigned]] = 1
        y_values.append(intensities[region])
        region_list.append(region)

    X = pd.DataFrame(design_array, index=region_list, columns=alt_alleles)
    y = pd.Series(y_values, index=region_list)

    # [Step 5] Covariates
    if include_covariates:
        kmer_pos = {
            region: region_lookup[allele][region]
            for allele in all_alleles
            for region in region_lookup[allele]
            if region in region_list
        }
        lp, sl = extract_covariates(region_list, kmer_pos)
        X["lp"] = lp
        X["sl"] = sl

    # [Step 6] Regression
    X_const = sm.add_constant(X)
    if model_type == "ols":
        model = sm.OLS(y.apply(math.log1p), X_const)
    else:
        model = sm.GLM(y, X_const, family=sm.families.NegativeBinomial())

    results = model.fit()

    # [Step 7] Output rows
    seen = set()
    for allele in alt_alleles:
        coef = results.params.get(allele, math.nan)
        pval = results.pvalues.get(allele, math.nan)
        for wildcard_kmer, filled_kmer, _ in allele_matched_kmers[motif_pos][snv_index].get(allele, []):
            key = (wildcard_kmer, filled_kmer)
            if key not in seen:
                seen.add(key)
                rows.append([
                    wildcard_kmer,
                    filled_kmer,
                    motif_pos,
                    snv_index,
                    "AFF",
                    allele,
                    coef,
                    pval,
                    motif_pos + snv_index,
                ])

    # [Step 8] Add ref allele with NA
    for wildcard_kmer, filled_kmer, _ in allele_matched_kmers[motif_pos][snv_index].get(ref_allele, []):
        key = (wildcard_kmer, filled_kmer)
        if key not in seen:
            seen.add(key)
            rows.append([
                wildcard_kmer,
                filled_kmer,
                motif_pos,
                snv_index,
                "AFF",
                ref_allele,
                "NA",
                "NA",
                motif_pos + snv_index,
            ])

    print(f"[✓] Regression done for window {motif_pos}:{snv_index} in {time.time() - start:.2f}s")
    return rows


In [81]:
import re
from collections import defaultdict
import random
import numpy as np
import statsmodels.api as sm
from pyfaidx import Fasta
import warnings
from statsmodels.discrete.discrete_model import NegativeBinomial
import hashlib
import pandas as pd
import math
import matplotlib.pyplot as plt
import sys
from utils import (
    extract_covariates
)

motif_pos = 0
snv_index = 0

rows = run_snv_regression(
    motif_pos,
    snv_index,
    allele_region_offsets,
    allele_matched_kmers,
    intensities,
    region_seq=region_seq,
    include_covariates = True
)


for row in rows:
    print(row)

[✓] Regression done for window 0:0 in 0.43s
['.GCCCGGA', 'AGCCCGGA', 0, 0, 'AFF', 'A', -0.15815798395206887, 1.088533115959818e-06, 0]
['.GCCCGGA', 'CGCCCGGA', 0, 0, 'AFF', 'C', 0.1061041676930918, 0.0020268882657664593, 0]
['.GCCCGGA', 'TGCCCGGA', 0, 0, 'AFF', 'T', -0.2171485557948205, 7.48926593570001e-10, 0]
['.GCCCGGA', 'GGCCCGGA', 0, 0, 'AFF', 'G', 'NA', 'NA', 0]


In [ ]:
all_rows = []

for motif_pos in sorted(allele_region_offsets):
    for snv_index in sorted(allele_region_offsets[motif_pos]):
        rows = run_snv_regression(
            motif_pos,
            snv_index,
            allele_region_offsets,
            allele_matched_kmers,
            intensities,
            region_seq=region_seq,
            include_covariates = True
        )
        all_rows.extend(rows)
        
print_rows_as_tsv(all_rows)
plot_aff_motif_effects(all_rows, "cov3.png")

In [82]:
all_rows = []

for motif_pos in sorted(allele_region_offsets):
    for snv_index in sorted(allele_region_offsets[motif_pos]):
        rows = run_snv_regression(
            motif_pos,
            snv_index,
            allele_region_offsets,
            allele_matched_kmers,
            intensities,
            region_seq=region_seq,
            include_covariates = True
        )
        all_rows.extend(rows)
        
print_rows_as_tsv(all_rows)
plot_aff_motif_effects(all_rows, "cov3.png")

[✓] Regression done for window 0:0 in 0.43s
[✓] Regression done for window 0:1 in 0.52s
[✓] Regression done for window 0:2 in 0.39s
[✓] Regression done for window 0:3 in 0.42s
[✓] Regression done for window 0:4 in 2.68s
[✓] Regression done for window 0:5 in 1.52s
[✓] Regression done for window 0:6 in 0.30s
[✓] Regression done for window 0:7 in 1.13s
[✓] Regression done for window 1:0 in 0.83s
[✓] Regression done for window 1:1 in 0.56s
[✓] Regression done for window 1:2 in 0.64s
[✓] Regression done for window 1:3 in 4.29s
[✓] Regression done for window 1:4 in 2.81s
[✓] Regression done for window 1:5 in 0.51s
[✓] Regression done for window 1:6 in 1.10s
[✓] Regression done for window 1:7 in 0.43s
[✓] Regression done for window 2:0 in 0.60s
[✓] Regression done for window 2:1 in 0.69s
[✓] Regression done for window 2:2 in 5.25s
[✓] Regression done for window 2:3 in 3.13s
[✓] Regression done for window 2:4 in 0.89s
[✓] Regression done for window 2:5 in 0.94s
[✓] Regression done for window 2

KeyboardInterrupt: 

In [16]:
def plot_aff_motif_effects(rows, save_path):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    import numpy as np
    import pandas as pd

    df = pd.DataFrame(
        rows,
        columns=[
            "wildcard_kmer",
            "filled_kmer",
            "window_index",
            "snp_index",
            "type",
            "allele",
            "coef",
            "pval",
            "absolute_pos",
        ],
    )

    df = df[df["type"] == "AFF"].copy()
    df["coef"] = pd.to_numeric(df["coef"], errors="coerce")
    df["pval"] = pd.to_numeric(df["pval"], errors="coerce")
    df = df.dropna(subset=["coef", "pval"])

    # Fix infinite values for -log10(pval)
    df["-log10(pval)"] = -np.log10(df["pval"].replace(0, np.nextafter(0, 1)))
    df["-log10(pval)"] = df["-log10(pval)"].clip(upper=300)

    df["absolute_position"] = df["window_index"] + df["snp_index"] + 1
    region_length = df["absolute_position"].max()

    allele_colors = {"A": "black", "C": "red", "G": "green", "T": "blue"}

    fig_width = max(12, region_length * 0.15)
    fig, axs = plt.subplots(2, 1, figsize=(fig_width, 6), sharex=True)

    for metric, ax in zip(["coef", "-log10(pval)"], axs):
        # print(f"[Debug] Plotting metric: {metric}")
        # print(f"[Debug] Any NaNs in '{metric}'? {df[metric].isna().sum()} / {len(df)}")
        # print(f"[Debug] Sample values for {metric}:\n{df[metric].head()}")

        for _, row in df.iterrows():
            x = int(row["absolute_position"])
            y = row[metric]
            allele = row["allele"]
            color = allele_colors.get(allele, "gray")
            if pd.notna(y):
                ax.text(
                    x,
                    y,
                    allele,
                    color=color,
                    fontsize=12,
                    ha="center",
                    va="center",
                    fontweight="bold",
                )

        ymin = df[metric].min()
        ymax = df[metric].max()
        yrange = ymax - ymin if ymax != ymin else 1
        padding = yrange * 0.1
        # print(f"[Debug] ymin: {ymin}, ymax: {ymax}, padding: {padding}")
        ax.set_ylim(ymin - padding, ymax + padding)
        ax.set_ylabel(metric)
        ax.grid(True, linestyle="--", alpha=0.4)

    axs[1].set_xlabel("Genomic Position")
    axs[0].set_title("Motif Scan: Coefficients and -log10(p-values)")

    step = 10 if region_length > 80 else 5 if region_length > 40 else 1
    axs[1].set_xticks(range(1, region_length + 1, step))
    axs[1].set_xlim(0.5, region_length + 0.5)

    plt.tight_layout()
    plt.savefig(save_path, dpi=300)
    plt.close()
    print(f"[Info] Figure saved to: {save_path}")



plot_aff_motif_effects(all_rows, "cov2.png")

[Info] Figure saved to: cov2.png


In [220]:
from collections import defaultdict
import numpy as np
import pandas as pd

def summarize_with_kmers(all_rows, allele_region_offsets, intensities):
    summary = []
    for row in all_rows:
        wildcard_kmer, filled_kmer, motif_pos, snv_index, _, allele, coef, pval, _ = row

        try:
            regions = allele_region_offsets[motif_pos][snv_index][allele]
        except KeyError:
            regions = {}

        region_intensities = [intensities[r] for r in regions if r in intensities]
        n_probes = len(region_intensities)
        avg_intensity = np.mean(region_intensities) if region_intensities else float("nan")

        summary.append({
            "motif_pos": motif_pos,
            "snv_index": snv_index,
            "allele": allele,
            "wildcard_kmer": wildcard_kmer,
            "filled_kmer": filled_kmer,
            "coef": coef,
            "pval": pval,
            "n_probes": n_probes,
            "avg_intensity": avg_intensity,
        })

    return pd.DataFrame(summary)

# Run it
summary_df = summarize_with_kmers(all_rows, allele_region_offsets, intensities)
with pd.option_context('display.max_rows', None, 'display.max_columns', None):
    display(summary_df)

,motif_pos,snv_index,allele,wildcard_kmer,filled_kmer,coef,pval,n_probes,avg_intensity
0,0,0,A,.CGGAGGG,ACGGAGGG,0.181141,0.0,1933,8.290216
1,0,0,G,.CGGAGGG,GCGGAGGG,0.23701,0.0,4757,8.952825
2,0,0,T,.CGGAGGG,TCGGAGGG,0.241546,0.0,1631,8.394205
3,0,0,C,.CGGAGGG,CCGGAGGG,NA,NA,3435,8.930937
4,0,1,A,C.GGAGGG,CAGGAGGG,-0.013323,0.256811,9879,4.694200
5,0,1,G,C.GGAGGG,CGGGAGGG,0.27043,0.0,5774,8.303923
6,0,1,T,C.GGAGGG,CTGGAGGG,0.025578,0.03126,9588,5.070135
7,0,1,C,C.GGAGGG,CCGGAGGG,NA,NA,3435,8.930937
8,0,2,A,CC.GAGGG,CCAGAGGG,-0.031097,0.009885,9400,4.325063
9,0,2,C,CC.GAGGG,CCCGAGGG,0.237399,0.0,3906,7.956764


In [205]:
# Define your target kmers
target_kmers = ["CCGGAAGG", "CGGAGCGG"]

# Initialize results
avg_intensities = {}

for kmer in target_kmers:
    if kmer not in kmers:
        print(f"[Warning] Kmer not found: {kmer}")
        continue

    region_intensities = []
    
    for region in kmers[kmer]:
        if region in intensities:
            region_intensities.append(intensities[region])
    
    if region_intensities:
        avg = sum(region_intensities) / len(region_intensities)
        avg_intensities[kmer] = avg
        print(f"{kmer} → Avg Intensity: {avg:.4f} over {len(region_intensities)} regions")
    else:
        print(f"{kmer} → No regions with matching intensity found.")

# Optional: print the full dictionary
# pprint(avg_intensities)


CCGGAAGG → Avg Intensity: 12.8074 over 1103 regions
CGGAGCGG → Avg Intensity: 11.0240 over 1511 regions


In [206]:
# Get region sets for both kmers
kmer1 = "CCGGAAGG"
kmer2 = "CGGAGCGG"

regions_kmer1 = set(kmers.get(kmer1, {}).keys())
regions_kmer2 = set(kmers.get(kmer2, {}).keys())

# Find intersection
overlap = regions_kmer1 & regions_kmer2

print(f"Total regions for {kmer1}: {len(regions_kmer1)}")
print(f"Total regions for {kmer2}: {len(regions_kmer2)}")
print(f"Number of overlapping regions: {len(overlap)}")

# Show overlapping region names (first 10, optionally print all)
if overlap:
    print("\nExample overlapping regions:")
    for region in list(overlap)[:10]:
        print(f"  {region}")
else:
    print("No overlapping regions.")


Total regions for CCGGAAGG: 1103
Total regions for CGGAGCGG: 1511
Number of overlapping regions: 53

Example overlapping regions:
  chr9:136050957-136052845
  chr16:2473440-2475256
  chr17:29566593-29569268
  chr6:31497606-31498628
  chr19:32405056-32406388
  chr3:167734493-167735871
  chr8:42541512-42542427
  chr19:36114753-36116108
  chrX:1391550-1391913
  chr22:21383660-21385589


In [208]:
# Remove overlapping regions from each set
unique_kmer1 = regions_kmer1 - overlap
unique_kmer2 = regions_kmer2 - overlap

# Compute new average intensity for each k-mer
def average_intensity(region_set):
    values = [intensities[r] for r in region_set if r in intensities]
    avg = np.mean(values) if values else float('nan')
    return avg, len(values)

avg1, count1 = average_intensity(unique_kmer1)
avg2, count2 = average_intensity(unique_kmer2)

print(f"{kmer1} → Avg Intensity (non-overlapping): {avg1:.4f} over {count1} regions")
print(f"{kmer2} → Avg Intensity (non-overlapping): {avg2:.4f} over {count2} regions")


CCGGAAGG → Avg Intensity (non-overlapping): 12.6317 over 1050 regions
CGGAGCGG → Avg Intensity (non-overlapping): 10.8326 over 1458 regions


In [129]:
from collections import defaultdict
import pandas as pd
import statsmodels.api as sm
import math
import numpy as np

# -- Run once for both cases --
motif_pos = 0
snv_index = 0

# Store output
results_summary = {}

for cov_flag in [True, False]:
    label = "WITH Covariates" if cov_flag else "WITHOUT Covariates"

    print(f"\n===== [{label}] =====")

    rows = run_snv_regression(
        motif_pos,
        snv_index,
        allele_region_offsets,
        allele_matched_kmers,
        intensities,
        region_seq=region_seq,
        include_covariates=cov_flag
    )

    # Extract modeled rows (exclude ref allele rows)
    modeled = [r for r in rows if r[6] != "NA"]

    # Show first few modeled results
    df_rows = pd.DataFrame(
        modeled,
        columns=[
            "wildcard_kmer", "filled_kmer", "window_index", "snp_index",
            "type", "allele", "coef", "pval", "absolute_pos"
        ]
    )
    display(df_rows.head())

    # Rebuild design matrix to compare internal stats
    all_alleles = allele_region_offsets[motif_pos][snv_index]
    ref_allele = region_seq[motif_pos + snv_index]
    alleles = {a: v for a, v in all_alleles.items() if a != ref_allele}

    all_regions = set()
    for region_dict in alleles.values():
        all_regions.update(region_dict.keys())

    design = []
    y_values = []
    region_list = []

    for region in all_regions:
        row = {a: 0 for a in alleles}
        for a in alleles:
            if region in alleles[a]:
                row[a] = 1
        if sum(row.values()) == 1 and region in intensities:
            design.append(row)
            y_values.append(intensities[region])
            region_list.append(region)

    X = pd.DataFrame(design)
    y = pd.Series(y_values, index=region_list)
    X.index = y.index

    if cov_flag:
        kmer_pos = {
            region: allele_region_offsets[motif_pos][snv_index][allele][region]
            for allele in alleles
            for region in allele_region_offsets[motif_pos][snv_index][allele]
        }
        lp, sl = extract_covariates(region_list, kmer_pos)
        X["lp"] = lp
        X["sl"] = sl

    if include_covariates:
        X_const = sm.add_constant(X)
    else:
        X_const = X  # no intercept

    model = sm.GLM(y, X_const, family=sm.families.NegativeBinomial())
    fit = model.fit()

    print(f"\nDesign matrix shape: {X.shape}")
    print(f"Condition number: {np.linalg.cond(X_const.values):.2f}")

    print("\nFitted coefficients:")
    display(fit.params)

    results_summary[label] = fit.params



===== [WITH Covariates] =====


,wildcard_kmer,filled_kmer,window_index,snp_index,type,allele,coef,pval,absolute_pos
0,.CGGAGGG,ACGGAGGG,0,0,AFF,A,0.181141,2.017258e-14,0
1,.CGGAGGG,GCGGAGGG,0,0,AFF,G,0.237010,1.805632e-33,0
2,.CGGAGGG,TCGGAGGG,0,0,AFF,T,0.241546,1.564239e-21,0



Design matrix shape: (7037, 5)
Condition number: 6582.95

Fitted coefficients:


A     0.840838
G     0.896707
T     0.901243
lp   -0.052512
sl    0.001173
dtype: float64


===== [WITHOUT Covariates] =====


,wildcard_kmer,filled_kmer,window_index,snp_index,type,allele,coef,pval,absolute_pos
0,.CGGAGGG,ACGGAGGG,0,0,AFF,A,-1.824485e+12,0.67984,0
1,.CGGAGGG,GCGGAGGG,0,0,AFF,G,-1.824485e+12,0.67984,0
2,.CGGAGGG,TCGGAGGG,0,0,AFF,T,-1.824485e+12,0.67984,0



Design matrix shape: (7037, 3)
Condition number: 1.80

Fitted coefficients:


A    2.023958
G    2.178565
T    2.083533
dtype: float64

In [130]:
print("==== X.head() ====")
display(X.head())

print("==== Column sums ====")
print(X.sum())

print("==== y.describe() ====")
print(y.describe())

print("==== Any NaNs in y? ====")
print(y.isna().sum())

print("==== Any duplicates in index? ====")
print(y.index.duplicated().sum())

print("==== Unique rows in X ====")
print(X.drop_duplicates().shape[0])


==== X.head() ====


,A,G,T
chr9:127555664-127556458,1,0,0
chr4:99950177-99950959,0,1,0
chr5:77077007-77077931,0,1,0
chr5:58459388-58461208,0,1,0
chr11:61967120-61968101,1,0,0


==== Column sums ====
A    1566
G    4182
T    1289
dtype: int64
==== y.describe() ====
count    7037.000000
mean        8.405330
std        14.782831
min         0.000000
25%         2.192540
50%         3.249620
75%         6.093920
max       156.466492
dtype: float64
==== Any NaNs in y? ====
0
==== Any duplicates in index? ====
0
==== Unique rows in X ====
3


In [121]:
from collections import Counter

def test_run_snv_regression_debug():
    motif_pos = 2
    snv_index = 0

    # Run regression
    rows = run_snv_regression(
        motif_pos,
        snv_index,
        allele_region_offsets,
        allele_matched_kmers,
        intensities,
        region_seq=region_seq,
        include_covariates=False,
    )

    # Check reference allele
    ref_allele = region_seq[motif_pos + snv_index]
    print(f"[Info] Reference allele at pos {motif_pos + snv_index}: {ref_allele}")

    # Check alleles being modeled
    all_alleles = list(allele_region_offsets[motif_pos][snv_index].keys())
    modeled_alleles = [a for a in all_alleles if a != ref_allele]
    print(f"[Info] All alleles: {all_alleles}")
    print(f"[Info] Modeled alleles (excluding ref): {modeled_alleles}")

    # Count how many times each region appears per allele
    region_counts = defaultdict(int)
    for allele in modeled_alleles:
        for region in allele_region_offsets[motif_pos][snv_index][allele]:
            region_counts[region] += 1

    ambiguous = [r for r, count in region_counts.items() if count > 1]
    print(f"[Info] Number of ambiguous regions (hit >1 allele): {len(ambiguous)}")
    print(f"[Info] Example ambiguous regions: {ambiguous[:5]}")

    # Manually build design matrix
    all_regions = set(region_counts)
    design = []
    y_values = []
    region_list = []

    for region in all_regions:
        row = {a: 0 for a in modeled_alleles}
        for a in modeled_alleles:
            if region in allele_region_offsets[motif_pos][snv_index][a]:
                row[a] = 1

        # Must hit only one allele and be present in intensities
        if sum(row.values()) == 1 and region in intensities:
            design.append(row)
            y_values.append(intensities[region])
            region_list.append(region)

    print(f"[Info] Final usable regions: {len(region_list)}")
    print(f"[Info] Example design row:\n{design[0] if design else 'None'}")

    # Display matrix
    X = pd.DataFrame(design)
    X["intensity"] = y_values
    display(X.head())

    # Print results
    print(f"[Info] First few regression rows:")
    for row in rows[:5]:
        print(row)

# Run the test
test_run_snv_regression_debug()

[Info] Reference allele at pos 2: G
[Info] All alleles: ['A', 'C', 'G', 'T']
[Info] Modeled alleles (excluding ref): ['A', 'C', 'T']
[Info] Number of ambiguous regions (hit >1 allele): 662
[Info] Example ambiguous regions: ['chr1:107140901-107142430', 'chr1:10793646-10794659', 'chr1:113811517-113812742', 'chr1:1406391-1408070', 'chr1:147541148-147541772']
[Info] Final usable regions: 9792
[Info] Example design row:
{'A': 1, 'C': 0, 'T': 0}


,A,C,T,intensity
0,1,0,0,1.15199
1,0,0,1,2.26794
2,0,0,1,1.40099
3,1,0,0,3.28847
4,1,0,0,4.21287


[Info] First few regression rows:
['.GAGGGGG', 'AGAGGGGG', 2, 0, 'AFF', 'A', 0.22025079905177616, 7.039796597525242e-53, 2]
['.GAGGGGG', 'CGAGGGGG', 2, 0, 'AFF', 'C', 0.786749069117356, 0.0, 2]
['.GAGGGGG', 'TGAGGGGG', 2, 0, 'AFF', 'T', 0.3748048410127507, 1.7475247835115403e-122, 2]
['.GAGGGGG', 'GGAGGGGG', 2, 0, 'AFF', 'G', 'NA', 'NA', 2]


# New section for normal EUBAR

In [2]:
from utils import (
    read_intensities,
    read_unique_kmer_positions,
    parse_snv_string,
    normalize_snv_region,
    get_snv_aligned_wildcards,
    # match_snv_aligned_kmers,
    run_per_motif_regression,
    sample_rand_kmers_per_allele,
    run_rand_regression_from_region_map,
    print_motif_effect_table
)

# genome =  "/home/aki/repos/INVPBM/rust_INVPBM/data/genome/hg38_ucsc.fa"
genome = "../../../../../d/OneDrive - McGill University/repos/hg38_ucsc.fa"
intensities = read_intensities("data/intensities/GABPA_MCF7_probeIntensity.bed")
kmers = read_unique_kmer_positions("data/array/MCF7_Array_8mer_ATAC.txt")
# intensities = read_intensities("data/intensities/DNase_GABPA_MCF7_ENCFF835KCG_probeIntensity.bed")
# kmers = read_unique_kmer_positions("data/array/MCF7_Array_8mer_DNase.txt")
mode = "neg-binomial"
kmer_size=8

/home/aki/miniconda3/lib/python3.8/site-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.24.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


In [4]:
snv_list =  "chr5:1295113:G>A"

In [5]:
from utils import parse_snv_string, normalize_snv_region

snv_str = "chr5:1295113:G>A"
chrom, snv_pos, ref, alt = parse_snv_string(snv_str)

snv_info = normalize_snv_region(
    chrom, snv_pos, ref, alt, genome, kmer_size, debug=True  # Set debug=False if you want clean output
)



Windowed region: chr5:1295106-1295120
Sequence: GCCCGGAGGGGGCTG
SNV should be at position 7 → base: G
[*] Reference allele matches genome — no reverse complement needed.

[returning]
  region_seq: GCCCGGAGGGGGCTG
  snv_index: 7
  ref_allele: G
  genome_base (final): G


In [115]:
def reverse_complement(seq):
    complement = str.maketrans("ACGTacgt", "TGCAtgca")
    return seq.translate(complement)[::-1]

def wildcard_match(kmer, wildcard):
    """
    Returns True if the kmer matches the wildcard pattern (dot as any base).
    Much faster than regex.
    """
    for k, w in zip(kmer, wildcard):
        if w != '.' and k != w:
            return False
    return True

def match_snv_aligned_kmers(snv_info, kmer_positions, kmer_size, include_revcomp=True):
    from collections import defaultdict

    def get_matches(wildcard, strand):
        region_to_alleles = defaultdict(set)
        region_kmer_hits = {}

        for kmer in kmer_positions:
            if len(kmer) != kmer_size:
                continue
            test_kmer = kmer if strand == "forward" else reverse_complement(kmer)
            if wildcard_match(test_kmer, wildcard):
                allele = test_kmer[wildcard.index(".")]
                for region_id, offset in kmer_positions[kmer].items():
                    region_to_alleles[region_id].add(allele)
                    region_kmer_hits.setdefault(region_id, {})[allele] = (offset, kmer)

        return region_to_alleles, region_kmer_hits

    allele_region_offsets = {}
    matched_regions = set()
    wildcards = get_snv_aligned_wildcards(snv_info, kmer_size)

    for motif_pos, wildcard in wildcards:
        fw_to_alleles, fw_hits = get_matches(wildcard, strand="forward")
        rc_to_alleles, rc_hits = get_matches(wildcard, strand="reverse") if include_revcomp else ({}, {})

        combined = defaultdict(set)
        for region, alleles in fw_to_alleles.items():
            combined[region].update(alleles)
        for region, alleles in rc_to_alleles.items():
            combined[region].update(alleles)

        # Now handle unambiguous
        for region, alleles in combined.items():
            if len(alleles) == 1:
                allele = next(iter(alleles))
                if region in fw_hits and allele in fw_hits[region]:
                    offset, _ = fw_hits[region][allele]
                elif region in rc_hits and allele in rc_hits[region]:
                    offset, _ = rc_hits[region][allele]
                else:
                    continue  # shouldn't happen, but safety

                allele_region_offsets.setdefault(motif_pos, {}).setdefault(allele, {})[region] = offset
                matched_regions.add(region)

    return allele_region_offsets, wildcards, matched_regions



# Step 2a: Get wildcards (e.g., .GGAGGG or G.GAGGG)
wildcards = get_snv_aligned_wildcards(snv_info, kmer_size)

# Step 2b: Match these wildcards to actual probe regions
allele_region_offsets, wildcards, matched_regions = match_snv_aligned_kmers(
    snv_info, kmers, kmer_size
)

# Summary of results
print("Wildcards:", wildcards)
print("Total matched regions (with RC):", len(matched_regions))
print("Alleles at motif_pos 0:")
for allele, regions in allele_region_offsets.get(2, {}).items():
    print(f"  {allele}: {len(regions)} regions")

Wildcards: [(0, 'GCCCGGA.'), (1, 'CCCGGA.G'), (2, 'CCGGA.GG'), (3, 'CGGA.GGG'), (4, 'GGA.GGGG'), (5, 'GA.GGGGC'), (6, 'A.GGGGCT'), (7, '.GGGGCTG')]
Total matched regions (with RC): 54163
Alleles at motif_pos 0:
  A: 1903 regions
  C: 805 regions
  G: 2913 regions
  T: 877 regions


In [116]:
# Testing the above function

wildcard = "CCGGA.GG"
kmer_size = 8  # just to be explicit
snv_index = wildcard.index(".")

# Step 1: Match forward k-mers
matching_kmers = [kmer for kmer in kmers if len(kmer) == kmer_size and wildcard_match(kmer, wildcard)]

print(f"[Step 1] Matching k-mers for wildcard {wildcard}:")
for kmer in matching_kmers:
    print(" ", kmer)

from collections import defaultdict

# Step 2: Region counts by allele (from forward strand)
allele_to_regions = defaultdict(set)

for kmer in matching_kmers:
    allele = kmer[snv_index]
    for region_id in kmers[kmer]:
        allele_to_regions[allele].add(region_id)

print("\n[Step 2] Region counts by allele:")
for allele, regions in allele_to_regions.items():
    print(f"  Allele {allele}: {len(regions)} regions")

# Step 3: Reverse complement region counts
print("\n[Step 3] Reverse complement k-mers and their region counts:")

rc_kmer_to_regions = {}

for kmer in matching_kmers:
    rc = reverse_complement(kmer)
    if rc in kmers:
        rc_kmer_to_regions[rc] = kmers[rc]
        print(f"  {rc}: {len(kmers[rc])} regions")
    else:
        print(f"  {rc}: 0 regions (not in kmers)")

# Step 4: Combine forward and reverse complement region counts
print("\n[Step 4] Combined region counts per forward k-mer (including reverse complements):")
for fwd_kmer in matching_kmers:
    rc_kmer = reverse_complement(fwd_kmer)
    allele = fwd_kmer[snv_index]

    fwd_regions = allele_to_regions[allele]
    rc_regions = kmers.get(rc_kmer, {})  # Dict of region_id → offset

    combined_regions = set(fwd_regions)
    if rc_kmer in kmers:
        combined_regions.update(rc_regions.keys())

    print(f"  {fwd_kmer}: {len(fwd_regions)} + {len(rc_regions)} = {len(combined_regions)} regions")

# Step 5: Count unambiguous regions per allele
region_to_alleles = defaultdict(set)

for fwd_kmer in matching_kmers:
    allele = fwd_kmer[snv_index]
    rc_kmer = reverse_complement(fwd_kmer)

    for region_id in kmers.get(fwd_kmer, {}):
        region_to_alleles[region_id].add(allele)

    for region_id in kmers.get(rc_kmer, {}):
        region_to_alleles[region_id].add(allele)  # map RC back to forward allele

unambiguous_by_allele = defaultdict(set)

for region_id, alleles in region_to_alleles.items():
    if len(alleles) == 1:
        allele = next(iter(alleles))
        unambiguous_by_allele[allele].add(region_id)

print("\n[Step 5] Unambiguous region counts per allele:")
for allele in sorted(unambiguous_by_allele.keys()):
    print(f"  Allele {allele}: {len(unambiguous_by_allele[allele])} unambiguous regions")


[Step 1] Matching k-mers for wildcard CCGGA.GG:
  CCGGAAGG
  CCGGACGG
  CCGGAGGG
  CCGGATGG

[Step 2] Region counts by allele:
  Allele A: 1103 regions
  Allele C: 574 regions
  Allele G: 1827 regions
  Allele T: 531 regions

[Step 3] Reverse complement k-mers and their region counts:
  CCTTCCGG: 1221 regions
  CCGTCCGG: 577 regions
  CCCTCCGG: 1738 regions
  CCATCCGG: 568 regions

[Step 4] Combined region counts per forward k-mer (including reverse complements):
  CCGGAAGG: 1103 + 1221 = 2283 regions
  CCGGACGG: 574 + 577 = 1128 regions
  CCGGAGGG: 1827 + 1738 = 3435 regions
  CCGGATGG: 531 + 568 = 1094 regions

[Step 5] Unambiguous region counts per allele:
  Allele A: 1903 unambiguous regions
  Allele C: 805 unambiguous regions
  Allele G: 2913 unambiguous regions
  Allele T: 877 unambiguous regions


In [185]:
def run_aff_regression(
    motif_pos,
    snv_index,
    allele_region_offsets,
    allele_matched_kmers,
    intensities,
    region_seq=None,
    model_type="nb",
    include_covariates=True,
    return_matrix=False
):
    import pandas as pd
    import numpy as np
    import statsmodels.api as sm
    import math
    from utils import extract_covariates

    rows = []
    ref_allele = region_seq[motif_pos + snv_index]
    alleles = list(allele_region_offsets.get(motif_pos, {}).keys())
    alt_alleles = [a for a in alleles if a != ref_allele]

    all_regions = set()
    for a in alleles:
        all_regions.update(allele_region_offsets[motif_pos][a].keys())

    design = []
    y_values = []
    region_list = []

    for region in all_regions:
        row = {a: 0 for a in alt_alleles}
        # Count how many alleles (alt + ref) this region maps to
        present_alleles = [a for a in alleles if region in allele_region_offsets[motif_pos][a]]
        if len(present_alleles) == 1 and region in intensities:
            only_allele = present_alleles[0]
            if only_allele in alt_alleles:
                row[only_allele] = 1
            # Else, it's the ref allele → all zeros
            design.append(row)
            y_values.append(intensities[region])
            region_list.append(region)


    if not design:
        return []

    X = pd.DataFrame(design, index=region_list)
    y = pd.Series(y_values, index=region_list)

    if include_covariates:
        kmer_pos = {
            region: allele_region_offsets[motif_pos][a][region]
            for a in alleles
            for region in allele_region_offsets[motif_pos][a]
        }
        lp, sl = extract_covariates(region_list, kmer_pos)
        X["lp"] = lp
        X["sl"] = sl

    X_const = sm.add_constant(X)
    model = sm.GLM(y, X_const, family=sm.families.NegativeBinomial()) if model_type == "nb" else sm.OLS(np.log1p(y), X_const)
    results = model.fit()

    seen = set()
    for a in alt_alleles:
        coef = results.params.get(a, math.nan)
        pval = results.pvalues.get(a, math.nan)
        for wildcard_kmer, filled_kmer, _ in allele_matched_kmers[motif_pos][snv_index].get(a, []):
            key = (wildcard_kmer, filled_kmer)
            if key in seen:
                continue
            seen.add(key)
            rows.append({
                "wildcard_kmer": wildcard_kmer,
                "filled_kmer": filled_kmer,
                "motif_pos": motif_pos,
                "snp_index": snv_index,
                "type": "AFF",
                "allele": a,
                "coef": coef,
                "pval": pval,
                "absolute_pos": motif_pos + snv_index,
            })


    # Add ref allele rows
    for wildcard_kmer, filled_kmer, _ in allele_matched_kmers[motif_pos][snv_index].get(ref_allele, []):
        key = (wildcard_kmer, filled_kmer)
        if key in seen:
            continue
        seen.add(key)
        rows.append({
            "wildcard_kmer": wildcard_kmer,
            "filled_kmer": filled_kmer,
            "motif_pos": motif_pos,
            "snp_index": snv_index,
            "type": "AFF",
            "allele": ref_allele,
            "coef": "NA",
            "pval": "NA",
            "absolute_pos": motif_pos + snv_index,
        })

    return (rows, X, y) if return_matrix else rows


In [186]:
allele_region_offsets, wildcards, matched_regions = match_snv_aligned_kmers(
    snv_info, kmers, kmer_size
)

# Match new format
allele_matched_kmers = {}
for motif_pos, wildcard in wildcards:
    snv_index = wildcard.index(".")
    allele_matched_kmers.setdefault(motif_pos, {}).setdefault(snv_index, {})
    for kmer in kmers:
        if wildcard_match(kmer, wildcard):
            allele = kmer[snv_index]
            for region_id in kmers[kmer]:
                allele_matched_kmers[motif_pos][snv_index].setdefault(allele, []).append(
                    (wildcard, kmer, region_id)
                )


# TEST 

In [226]:
from utils import (
    read_intensities,
    read_unique_kmer_positions,
    parse_snv_string,
    normalize_snv_region,
    get_snv_aligned_wildcards,
    # match_snv_aligned_kmers,
    run_per_motif_regression,
    sample_rand_kmers_per_allele,
    run_rand_regression_from_region_map,
    print_motif_effect_table
)
all_results = []
snvs = ["chr5:1295113:G>A", "chr5:1295135:G>A"]
rand_n=50000

for snv_str in snvs:
    try:
        chrom, snv_pos, ref, alt = parse_snv_string(snv_str)
        snv_info = normalize_snv_region(chrom, snv_pos, ref, alt, genome, kmer_size)
        snv_info["snv_str"] = snv_str
        snv_info["wildcards"] = dict(get_snv_aligned_wildcards(snv_info, kmer_size))

        allele_region_offsets, wildcards, matched_regions = match_snv_aligned_kmers(
            snv_info, kmers, kmer_size
        )

        # Step 1: build allele_matched_kmers
        allele_matched_kmers = {}
        for motif_pos, wildcard in wildcards:
            snv_index = wildcard.index(".")
            allele_matched_kmers.setdefault(motif_pos, {}).setdefault(snv_index, {})
            for kmer in kmers:
                if wildcard_match(kmer, wildcard):
                    allele = kmer[snv_index]
                    for region_id in kmers[kmer]:
                        allele_matched_kmers[motif_pos][snv_index].setdefault(allele, []).append(
                            (wildcard, kmer, region_id)
                        )

        # Step 2: run AFF regression across motif
        results_aff = []
        for motif_pos, snv_dict in allele_matched_kmers.items():
            for snv_index in snv_dict:
                rows = run_aff_regression(
                    motif_pos=motif_pos,
                    snv_index=snv_index,
                    allele_region_offsets=allele_region_offsets,
                    allele_matched_kmers=allele_matched_kmers,
                    intensities=intensities,
                    region_seq=snv_info["region_seq"],
                )
                results_aff.extend(rows)

        # Step 3: RAND
        rand_regions_per_allele, _ = sample_rand_kmers_per_allele(
            kmers,
            matched_regions,
            snv_str=snv_info["snv_str"],
            kmer_size=kmer_size,
            rand_n=rand_n,
        )

        results_rand = run_rand_regression_from_region_map(
            rand_regions_per_allele=rand_regions_per_allele,
            probe_intensities=intensities,
            snv_str=snv_info["snv_str"],
        )

        # Step 4: Print results
        print_motif_effect_table(
            snv_str=snv_info["snv_str"],
            chrom=snv_info["chrom"],
            pos=snv_info["pos"],
            region_seq=snv_info["region_seq"],
            results=results_aff + results_rand,
        )

        all_results.extend(results_aff + results_rand)

    except Exception as e:
        print(f"[Error] {snv_str}: {e}")


chr5:1295113:G>A	chr5	1295113	GCCCGGAGGGGGCTG
AFF	chr5:1295113:G>A	A	0,1,2,3,4,5,6,7	0.4224229,0.810549,0.3949084,0.309578,0.02693133,0.03386981,0.1262536,-0.01335332	5.864666e-38,4.646503e-189,3.024848e-36,8.83873e-23,0.1549127,0.1217528,1.778396e-06,0.5367808
AFF	chr5:1295113:G>A	C	0,1,2,3,4,5,6,7	0.1421341,0.22052,0.1441168,0.07602095,0.2560922,0.1104585,0.2392032,0.193193	8.075435e-06,4.257986e-09,0.0006508364,0.05344881,2.986269e-19,0.0002221393,4.501545e-13,2.037447e-16
AFF	chr5:1295113:G>A	G	0,1,2,3,4,5,6,7	0,0,0,0,0,0,0,0	NA,NA,NA,NA,NA,NA,NA,NA
AFF	chr5:1295113:G>A	T	0,1,2,3,4,5,6,7	0.007938478,0.1034015,0.002447182,-0.0116785,-0.08252455,-0.18389,-0.05593623,-0.1150854	0.8473239,0.005975622,0.9534454,0.7663384,9.399777e-05,2.163591e-14,0.03662137,2.358714e-08
RAND	chr5:1295113:G>A	A	0,1,2,3,4,5,6,7	0.05014333,-0.01876925,-0.006218753,0.02068526,0.02726132,-0.03439344,0.02863303,0.007887638	0.1558002,0.5822444,0.8551024,0.5489603,0.4352677,0.3163126,0.403634,0.8169746
RAND	chr

In [215]:
def deterministic_hash(string, max_val=2**32):
    return int(hashlib.md5(string.encode()).hexdigest(), 16) % max_val
import hashlib

import random


def run_rand_regression_from_region_map(
    rand_regions_per_allele, 
    probe_intensities, 
    snv_str, 
    model_type="nb",
    return_matrix=False
):
    """
    Runs per-position regression using predefined allele-to-region mapping.

    Parameters:
        rand_regions_per_allele: dict of {motif_pos: {allele: set(region_ids)}}
        probe_intensities: dict of region -> intensity
        snv_str: e.g., 'chr5:1295113:C>T'
        model_type: 'nb' or 'ols'
        return_matrix: if True, also return (results, X_df, y)

    Returns:
        List of dicts with motif_pos, allele, coef, pval, n, label="RAND"
        Optionally returns X_df and y if return_matrix=True
    """
    import pandas as pd
    import numpy as np
    import statsmodels.api as sm

    results = []
    all_alleles = ["A", "C", "G", "T"]

    for motif_pos in sorted(rand_regions_per_allele):
        region_sets = rand_regions_per_allele[motif_pos]

        all_regions = set()
        for allele in all_alleles:
            all_regions.update(region_sets.get(allele, set()))

        y, lp_cov, sl_cov = [], [], []
        X_alleles = {a: [] for a in all_alleles}
        region_list = []

        for region in all_regions:
            if region not in probe_intensities:
                continue
            region_list.append(region)
            y.append(probe_intensities[region])

            try:
                _, coords = region.split(":")
                start, end = map(int, coords.split("-"))
                length = end - start
            except:
                length = 200
            offset = 1  # dummy for now

            lp_cov.append(offset / length)
            sl_cov.append(length)

            for allele in all_alleles:
                X_alleles[allele].append(1 if region in region_sets.get(allele, set()) else 0)

        if len(y) < 10:
            continue

        X_df = pd.DataFrame({a: X_alleles[a] for a in all_alleles}, index=region_list)
        X_df["lp"] = lp_cov
        X_df["sl"] = sl_cov
        X_const = sm.add_constant(X_df)
        y = np.array(y)

        if model_type == "ols":
            y_transformed = np.log1p(y)
            model = sm.OLS(y_transformed, X_const)
        else:
            model = sm.GLM(y, X_const, family=sm.families.NegativeBinomial())

        fit = model.fit()

        for allele in all_alleles:
            results.append({
                "snv_str": snv_str,
                "motif_pos": motif_pos,
                "allele": allele,
                "coef": fit.params.get(allele, 0.0),
                "pval": fit.pvalues.get(allele, np.nan),
                "n": len(region_sets.get(allele, [])),
                "label": "RAND"
            })

        if return_matrix:
            return results, X_df, y

    if return_matrix:
        return results, None, None
    return results


def sample_rand_kmers_per_allele(
    kmers,
    matched_regions,
    snv_str,
    kmer_size=8,
    rand_n=50000
):
    """
    Randomly sample k-mers, tracking alleles per motif position.
    Skips any regions used in AFF.

    Returns:
    - rand_regions_per_allele: {motif_pos: {allele: {region: offset}}}
    - used_regions: set of sampled region names
    """
    random.seed(deterministic_hash(snv_str))

    all_kmer_keys = list(kmers.keys())
    used_regions = set()
    rand_regions_per_allele = defaultdict(lambda: defaultdict(dict))

    attempts = 0
    max_attempts = rand_n * 20  # extra buffer

    while len(used_regions) < rand_n and attempts < max_attempts:
        kmer = random.choice(all_kmer_keys)
        region_dict = kmers[kmer]

        # Filter regions not in AFF and not already used
        available_regions = [r for r in region_dict if r not in matched_regions and r not in used_regions]
        if not available_regions:
            attempts += 1
            continue

        region = random.choice(available_regions)
        offset = region_dict[region]
        motif_pos = random.randint(0, kmer_size - 1)
        allele = kmer[motif_pos]

        rand_regions_per_allele[motif_pos][allele][region] = offset
        used_regions.add(region)
        attempts += 1

    if len(used_regions) < rand_n:
        print(f"[WARN] Only collected {len(used_regions)} usable regions for RAND.")

    return rand_regions_per_allele, used_regions


# Run the random regression
rand_regions_per_allele, _ = sample_rand_kmers_per_allele(
    kmers,
    matched_regions,
    snv_str=snv_info["snv_str"],
    kmer_size=kmer_size,
    rand_n=50000,  # adjust as needed
)

rand_rows, rand_X, rand_y = run_rand_regression_from_region_map(
    rand_regions_per_allele=rand_regions_per_allele,
    probe_intensities=intensities,
    snv_str=snv_info["snv_str"],
    model_type="nb",
    return_matrix=True,
)

# Print summary of the matrix
print("[RAND] Design matrix shape:", rand_X.shape)
print("[RAND] Sample design matrix:")
display(rand_X.head())

# Show rows where all alleles are 0 (if any)
allele_cols = [col for col in rand_X.columns if col in ["A", "C", "G", "T"]]
zero_rows = rand_X[allele_cols].sum(axis=1) == 0

print(f"\n[Info] Found {zero_rows.sum()} rows where all alleles are 0:")
display(rand_X[zero_rows])
print("\n[Corresponding y values]:")
display(rand_y[zero_rows])


[RAND] Design matrix shape: (6155, 6)
[RAND] Sample design matrix:


,A,C,G,T,lp,sl
chr2:152497742-152498238,0,1,0,0,0.002016,496
chr12:124913963-124914264,0,0,1,0,0.003322,301
chr20:22806859-22808165,1,0,0,0,0.000766,1306
chr2:28952069-28952357,0,0,1,0,0.003472,288
chr10:30581853-30582083,0,0,0,1,0.004348,230



[Info] Found 0 rows where all alleles are 0:


,A,C,G,T,lp,sl



[Corresponding y values]:


array([], dtype=float64)

In [216]:
from collections import Counter

for motif_pos, allele_dict in rand_regions_per_allele.items():
    print(f"\n[Motif Pos {motif_pos}]")
    for allele, regions in allele_dict.items():
        print(f"  Allele {allele}: {len(regions)} regions")



[Motif Pos 4]
  Allele C: 1616 regions
  Allele A: 1533 regions
  Allele T: 1502 regions
  Allele G: 1546 regions

[Motif Pos 7]
  Allele G: 1557 regions
  Allele T: 1542 regions
  Allele A: 1582 regions
  Allele C: 1586 regions

[Motif Pos 5]
  Allele T: 1528 regions
  Allele A: 1536 regions
  Allele C: 1613 regions
  Allele G: 1476 regions

[Motif Pos 3]
  Allele G: 1638 regions
  Allele A: 1623 regions
  Allele C: 1570 regions
  Allele T: 1545 regions

[Motif Pos 0]
  Allele G: 1552 regions
  Allele A: 1533 regions
  Allele C: 1542 regions
  Allele T: 1528 regions

[Motif Pos 1]
  Allele G: 1540 regions
  Allele C: 1559 regions
  Allele A: 1541 regions
  Allele T: 1575 regions

[Motif Pos 2]
  Allele A: 1649 regions
  Allele C: 1539 regions
  Allele T: 1556 regions
  Allele G: 1635 regions

[Motif Pos 6]
  Allele T: 1543 regions
  Allele G: 1585 regions
  Allele C: 1571 regions
  Allele A: 1559 regions


In [218]:
# All regions from all kmers (flattened)
all_regions = set()
for kmer in kmers:
    all_regions.update(kmers[kmer].keys())

# Remove any that were used for AFF
eligible_random_regions = all_regions - matched_regions

print(f"[INFO] Total regions in kmers: {len(all_regions)}")
print(f"[INFO] Matched (AFF) regions: {len(matched_regions)}")
print(f"[INFO] Eligible random regions: {len(eligible_random_regions)}")


[INFO] Total regions in kmers: 149216
[INFO] Matched (AFF) regions: 51735
[INFO] Eligible random regions: 97481


In [222]:
def filter_kmers_exclude_regions(kmers, matched_regions):
    """
    Returns a new kmers dictionary excluding any region that overlaps with matched_regions (AFF).
    """
    filtered_kmers = {}

    for kmer, region_dict in kmers.items():
        new_region_dict = {r: offset for r, offset in region_dict.items() if r not in matched_regions}
        if new_region_dict:
            filtered_kmers[kmer] = new_region_dict

    return filtered_kmers

def sample_rand_kmers_per_allele(
    kmers,
    matched_regions,
    snv_str,
    kmer_size=8,
    rand_n=5000
):
    """
    Randomly sample k-mers, tracking alleles per motif position.
    Skips any regions used in AFF by pre-filtering.

    Returns:
    - rand_regions_per_allele: {motif_pos: {allele: {region: offset}}}
    - used_regions: set of sampled region names
    """
    from collections import defaultdict
    import random

    random.seed(deterministic_hash(snv_str))
    filtered_kmers = filter_kmers_exclude_regions(kmers, matched_regions)

    all_kmer_keys = list(filtered_kmers.keys())
    used_regions = set()
    rand_regions_per_allele = defaultdict(lambda: defaultdict(dict))

    attempts = 0
    max_attempts = rand_n * 20

    while len(used_regions) < rand_n and attempts < max_attempts:
        kmer = random.choice(all_kmer_keys)
        region_dict = filtered_kmers[kmer]

        # Skip if all regions already used
        available_regions = [r for r in region_dict if r not in used_regions]
        if not available_regions:
            attempts += 1
            continue

        region = random.choice(available_regions)
        offset = region_dict[region]
        motif_pos = random.randint(0, kmer_size - 1)
        allele = kmer[motif_pos]

        rand_regions_per_allele[motif_pos][allele][region] = offset
        used_regions.add(region)
        attempts += 1

    if len(used_regions) < rand_n:
        print(f"[WARN] Only collected {len(used_regions)} usable regions for RAND.")

    return rand_regions_per_allele, used_regions


rand_regions_per_allele, _ = sample_rand_kmers_per_allele(
    kmers,
    matched_regions,
    snv_str=snv_info["snv_str"],
    kmer_size=kmer_size,
    rand_n=5000,  # adjust as needed
)

In [223]:
rand_rows, rand_X, rand_y = run_rand_regression_from_region_map(
    rand_regions_per_allele=rand_regions_per_allele,
    probe_intensities=intensities,
    snv_str=snv_info["snv_str"],
    model_type="nb",
    return_matrix=True,
)

# Print summary of the matrix
print("[RAND] Design matrix shape:", rand_X.shape)
print("[RAND] Sample design matrix:")
display(rand_X.head())

# Show rows where all alleles are 0 (if any)
allele_cols = [col for col in rand_X.columns if col in ["A", "C", "G", "T"]]
zero_rows = rand_X[allele_cols].sum(axis=1) == 0

print(f"\n[Info] Found {zero_rows.sum()} rows where all alleles are 0:")
display(rand_X[zero_rows])
print("\n[Corresponding y values]:")
display(rand_y[zero_rows])


[RAND] Design matrix shape: (584, 6)
[RAND] Sample design matrix:


,A,C,G,T,lp,sl
chr3:188838864-188839760,0,0,0,1,0.001116,896
chr15:89416119-89416843,0,1,0,0,0.001381,724
chr10:9301833-9301994,0,0,1,0,0.006211,161
chr2:152497742-152498238,0,1,0,0,0.002016,496
chr18:322523-322890,0,0,1,0,0.002725,367



[Info] Found 0 rows where all alleles are 0:


,A,C,G,T,lp,sl



[Corresponding y values]:


array([], dtype=float64)